[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Pagination &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API and the `Readings`
class. Run it first.


In [1]:
import importlib
import math
import sys
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()


class Readings:
    """The readings for a query, fetched a page at a time, and only when a loop asks for more."""

    def __init__(self, **params):
        self.params = params
        self.pages_fetched = 0

    def __iter__(self):
        response = requests.get(f"{BASE}/network/readings", params=self.params, timeout=10)
        while True:
            response.raise_for_status()
            self.pages_fetched += 1
            for reading in response.json()["readings"]:
                yield reading
            if "next" not in response.links:
                return
            response = requests.get(response.links["next"]["url"], timeout=10)


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A page in the middle of the list.


In [2]:
body = requests.get(f"{BASE}/network/readings", params={"page": 3, "per_page": 50}, timeout=10).json()

print(len(body["readings"]), "readings, from", body["readings"][0]["time"], "to", body["readings"][-1]["time"])


50 readings, from 2026-02-27T19:00Z to 2026-02-28T11:00Z


Pages 1 and 2 held the first 100 readings, so page 3 starts with the 101st: the Oslo reading in the
34th hour, because every hour holds three readings, Bergen's, Oslo's and Tromso's, in that order.


**2.** A number of pages, checked.


In [3]:
url = f"{BASE}/network/readings"
query = {"station": "oslo", "per_page": 25}
body = requests.get(url, params=query, timeout=10).json()
pages = math.ceil(body["total"] / body["per_page"])
print(pages, "pages")

for page in [pages, pages + 1]:
    readings = requests.get(url, params={**query, "page": page}, timeout=10).json()["readings"]
    print("page", page, "holds", len(readings), "readings")


3 pages
page 3 holds 22 readings
page 4 holds 0 readings


Oslo has 72 readings, and 72 divided by 25 is 2.88, which `ceil` rounds up to 3. The third page
holds the 22 left after two full pages, and the page after it is empty, which confirms that page 3
is the last.


**3.** The addresses a page names.


In [4]:
for page in [1, 4, 8]:
    response = requests.get(f"{BASE}/network/readings", params={"page": page}, timeout=10)
    print(page, list(response.links))


1 ['next', 'last']
4 ['prev', 'next', 'last', 'first']
8 ['prev', 'first']


The first page has nothing before it, so it names no `prev` and no `first`, and the last page has
nothing after it, so it names no `next` and no `last`. A page in the middle names all four.


**4.** Every page of one station, by link.


In [5]:
response = requests.get(f"{BASE}/network/readings", params={"station": "bergen", "per_page": 40}, timeout=10)
bergen, sent = response.json()["readings"], 1
while "next" in response.links:
    response = requests.get(response.links["next"]["url"], timeout=10)
    bergen += response.json()["readings"]
    sent += 1

warmest = max(reading["temperature_c"] for reading in bergen)
print(sent, "requests for", len(bergen), "readings")
print("warmest:", warmest, "°C, at", [reading["time"] for reading in bergen if reading["temperature_c"] == warmest])


2 requests for 72 readings
warmest: 4.8 °C, at ['2026-02-27T15:00Z', '2026-02-28T16:00Z']


72 readings at 40 a page make two pages, so two requests. Two readings share the warmest
temperature, so the answer lists the time of each, where taking a single reading would have hidden
the second.


**5.** Every event, by cursor.


In [6]:
events, cursor, pages = [], None, 0
while True:
    body = requests.get(f"{BASE}/network/events", params={"limit": 25, "cursor": cursor}, timeout=10).json()
    events += body["events"]
    pages += 1
    cursor = body["next_cursor"]
    if cursor is None:
        break

print(pages, "pages,", len(events), "events, the oldest:", events[-1]["summary"])


3 pages, 54 events, the oldest: thermometer and rain gauge calibrated


25 events a page need three pages for 54 events, the last holding four. The oldest event is Bergen's
calibration on October 14, 2025, the date its instruments' `last_calibrated` gives in `/network`.


**6.** A search that stops at the page it needs.


In [7]:
oslo = Readings(station="oslo", per_page=10)
cold = next(reading for reading in oslo if reading["temperature_c"] < -7)

print(cold)
print(oslo.pages_fetched, "pages fetched")


{'station': 'oslo', 'time': '2026-02-27T02:00Z', 'temperature_c': -7.1}
2 pages fetched


`next` took the first reading that the generator expression let through, and nothing asked for
another. That reading is the 17th, on page 2 at 10 a page, so the other six of Oslo's eight pages
were never fetched. The generator expression, from the **Comprehensions** notebook, asked `Readings`
for a reading at a time, and `Readings` asked the practice API for a page only when it had handed out
every reading of the page before.


---

&#8592; **Back to:** [Pagination](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/10-pagination.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
